<a href="https://colab.research.google.com/github/Aanoush-Surana/AutoEIT-Transcription/blob/main/AutoEIT_Transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai-whisper librosa soundfile noisereduce openpyxl jiwer pandas rapidfuzz
!apt-get install -q ffmpeg

import os, re, gc
import numpy as np
import pandas as pd
import librosa, soundfile as sf
import noisereduce as nr
import whisper
from jiwer import wer, cer
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from rapidfuzz import fuzz
from google.colab import files

DATA_DIR      = "/content/autoeit"
AUDIO_DIR     = os.path.join(DATA_DIR, "audio")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

uploaded = files.upload()

audio_xlsx_path   = None
scoring_xlsx_path = None
audio_files       = {}

for fname, data in uploaded.items():
    clean = re.sub(r" \(\d+\)", "", fname)
    if clean.endswith(".xlsx"):
        path = os.path.join(DATA_DIR, clean)
        with open(path, "wb") as f: f.write(data)
        if "audio" in clean.lower() or "transcribing" in clean.lower():
            audio_xlsx_path = path
        elif "scoring" in clean.lower() or "transcription" in clean.lower():
            scoring_xlsx_path = path
    elif clean.lower().endswith((".mp3", ".wav", ".m4a")):
        path = os.path.join(AUDIO_DIR, clean)
        with open(path, "wb") as f: f.write(data)
        pid = re.split(r"[_\-]", os.path.splitext(clean)[0])[0]
        audio_files[pid] = path
        print(f"  Audio loaded: {pid}")

print(f"\nAudio xlsx   : {audio_xlsx_path}")
print(f"Scoring xlsx : {scoring_xlsx_path}")

def preprocess(input_path, output_path):
    audio, _ = librosa.load(input_path, sr=16000, mono=True)
    noise    = audio[:16000] if len(audio) > 16000 else audio
    audio    = nr.reduce_noise(y=audio, sr=16000, y_noise=noise)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.7
    audio, _ = librosa.effects.trim(audio, top_db=20)
    sf.write(output_path, audio, 16000)
    return output_path

processed = {}
for pid, path in audio_files.items():
    out = os.path.join(PROCESSED_DIR, pid + ".wav")
    processed[pid] = preprocess(path, out)
    print(f"  Preprocessed: {pid}")

if 'model' in globals():
    del model; gc.collect()

MODEL_SIZE = "medium"
print(f"\nLoading whisper {MODEL_SIZE} on CPU...")
model = whisper.load_model(MODEL_SIZE, device="cpu")
print(f"Whisper {MODEL_SIZE} ready")

def clean_text(t):
    return re.sub(r"\[.*?\]", "", str(t)).lower().strip()

def merge_segments(segs, gap=1.5):

    if not segs: return segs
    m = [dict(segs[0])]
    for s in segs[1:]:
        if s["start"] - m[-1]["end"] < gap:
            m[-1]["text"] += " " + s["text"]
            m[-1]["end"]   = s["end"]
        else:
            m.append(dict(s))
    return m

def get_sentences(segs, prompts=None, n=30):
    segs    = merge_segments(segs)
    cleaned = [clean_text(s["text"]) for s in segs if s["text"].strip()]
    if prompts and len(prompts) == n:
        out = []
        for p in prompts:
            if cleaned:
                scores = [fuzz.partial_ratio(clean_text(p), c) for c in cleaned]
                out.append(cleaned[int(np.argmax(scores))])
            else:
                out.append("[no_response]")
        return out
    result = cleaned[:n]
    result += ["[no_response]"] * max(0, n - len(result))
    return result

all_segments = {}
for pid, path in processed.items():
    print(f"\n  Transcribing {pid}...")
    r = model.transcribe(
        path,
        language="es",
        task="transcribe",
        temperature=0,
        condition_on_previous_text=False,
        fp16=False,
    )
    all_segments[pid] = r["segments"]
    print(f"  {pid}: {len(r['segments'])} raw segments ")

audio_xls   = pd.ExcelFile(audio_xlsx_path)
scoring_xls = pd.ExcelFile(scoring_xlsx_path)

audio_sheets   = [s for s in audio_xls.sheet_names   if s != "Info"]
scoring_sheets = [s for s in scoring_xls.sheet_names if s != "Info"]

def pid_digits(s):
    return re.sub(r"[^0-9]", "", str(s)).lstrip("0")

sheet_map   = {}
stimuli_map = {}

for pid in audio_files:
    for sheet in audio_sheets:
        if pid_digits(sheet.split("-")[0]) == pid_digits(pid):
            sheet_map[pid]   = sheet
            df               = audio_xls.parse(sheet)
            stim_col         = next((c for c in df.columns
                                     if "stim" in str(c).lower()), df.columns[1])
            stimuli_map[pid] = df[stim_col].fillna("").astype(str).tolist()[:30]
            print(f"  Sheet matched: {pid} → '{sheet}'")
            break

scoring_map = {}
for sheet in scoring_sheets:
    df      = scoring_xls.parse(sheet)
    ref_col = next((c for c in df.columns
                    if any(k in str(c).lower()
                           for k in ["transcri", "rater", "human"])),
                   df.columns[2])
    trans   = df[ref_col].fillna("").astype(str).tolist()[:30]
    scoring_map[pid_digits(sheet.split("-")[0])] = (sheet, trans)

scoring_list = list(scoring_map.values())
demo_pairing = {}
for i, pid in enumerate(sorted(audio_files.keys())):
    demo_pairing[pid] = scoring_list[i % len(scoring_list)]
    print(f"  WER pairing:  {pid} → '{demo_pairing[pid][0]}' "
          f"(demo — different participant)")

asr_outputs = {}
for pid in audio_files:
    segs             = all_segments.get(pid, [])
    prompts          = stimuli_map.get(pid, [])
    asr_outputs[pid] = get_sentences(segs, prompts=prompts, n=30)
    scored = sum(1 for s in asr_outputs[pid] if s != "[no_response]")
    print(f"  {pid}: {scored}/30 sentences aligned")

print("Computing WER against Scoring xlsx references (cross-participant demo)")

results = {}
summary = []

for pid in audio_files:
    asr                  = asr_outputs[pid]
    scoring_sheet, human = demo_pairing[pid]
    res                  = []

    for i in range(30):
        h = clean_text(human[i]) if i < len(human) else ""
        a = asr[i] if i < len(asr) else "[no_response]"

        if h and a and a != "[no_response]":
            try:
                w = wer(h, a)
                c = cer(h, a)
            except Exception as e:
                print(f"  WER error sentence {i+1} ({pid}): {e}")
                w, c = None, None
        else:
            w, c = None, None

        res.append((a, h, w, c))

    results[pid] = (sheet_map.get(pid, "?"), scoring_sheet, res)

    valid = [(w, c) for (_, _, w, c) in res if w is not None]
    if valid:
        avg_w = sum(v[0] for v in valid) / len(valid)
        avg_c = sum(v[1] for v in valid) / len(valid)
        agree = (1 - avg_w) * 100
        summary.append((pid, sheet_map.get(pid,"?"), scoring_sheet,
                         avg_w, avg_c, agree, len(valid)))

print(f"\n{'='*78}")
print(f"{'PID':<10} {'Audio Sheet':<14} {'Ref Sheet':<14} "
      f"{'WER':>7} {'CER':>7} {'Agreement':>11} {'Scored':>8}")
print(f"{'='*78}")
for pid, asheet, rsheet, w, c, ag, n in summary:
    print(f"{pid:<10} {asheet:<14} {rsheet:<14} "
          f"{w:>7.3f} {c:>7.3f} {ag:>10.1f}% {n:>5}/30")
print(f"Model: Whisper {MODEL_SIZE} (CPU)")
print("  Cross-participant demo — WER reflects mismatch, not ASR accuracy")
print("   In production: matched human transcriptions give meaningful WER")

output_path = "/content/AutoEIT_GSoC_Output.xlsx"
wb          = load_workbook(audio_xlsx_path)

HEADER_FILL = PatternFill("solid", fgColor="1B4F8A")
ASR_FILL    = PatternFill("solid", fgColor="D6E4F0")
GOOD_FILL   = PatternFill("solid", fgColor="C6EFCE")
WARN_FILL   = PatternFill("solid", fgColor="FFEB9C")
BAD_FILL    = PatternFill("solid", fgColor="FFC7CE")
WHITE_FONT  = Font(bold=True, color="FFFFFF")

for pid, (audio_sheet, scoring_sheet, res) in results.items():
    if audio_sheet not in wb.sheetnames:
        print(f"  Skipping {pid} — sheet '{audio_sheet}' not in workbook")
        continue

    ws     = wb[audio_sheet]
    header = {ws.cell(1, c).value: c for c in range(1, ws.max_column + 1)}

    def get_or_create_col(name):
        if name not in header:
            idx            = ws.max_column + 1
            cell           = ws.cell(row=1, column=idx, value=name)
            cell.fill      = HEADER_FILL
            cell.font      = WHITE_FONT
            header[name]   = idx
        return header[name]

    asr_c  = get_or_create_col("ASR Transcription")
    ref_c  = get_or_create_col(f"Reference ({scoring_sheet})")
    wer_c  = get_or_create_col("WER")
    cer_c  = get_or_create_col("CER")
    agr_c  = get_or_create_col("Agreement %")

    for i, (asr_txt, ref_txt, w, c) in enumerate(res, start=2):
        ws.cell(row=i, column=asr_c, value=asr_txt).fill = ASR_FILL
        ws.cell(row=i, column=ref_c, value=ref_txt)

        if w is not None:
            agree = (1 - w) * 100
            fill  = GOOD_FILL if agree >= 80 else \
                    WARN_FILL if agree >= 60 else BAD_FILL
            ws.cell(row=i, column=wer_c, value=round(w, 4)).fill = fill
            ws.cell(row=i, column=cer_c, value=round(c, 4)).fill = fill
            ws.cell(row=i, column=agr_c, value=f"{agree:.1f}%").fill = fill
        else:
            for col_idx in [wer_c, cer_c, agr_c]:
                ws.cell(row=i, column=col_idx, value="N/A")

    for col_cells in ws.columns:
        max_len = max((len(str(cell.value or "")) for cell in col_cells), default=10)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 52)

if "Summary" in wb.sheetnames:
    del wb["Summary"]
ws_s = wb.create_sheet("Summary", 0)
hdrs = ["Participant", "Audio Sheet", "Reference Sheet", "Whisper Model",
        "WER", "CER", "Agreement %", "Sentences Scored", "Note"]
for ci, h in enumerate(hdrs, 1):
    cell      = ws_s.cell(row=1, column=ci, value=h)
    cell.fill = HEADER_FILL
    cell.font = WHITE_FONT

for ri, (pid, asheet, rsheet, w, c, ag, n) in enumerate(summary, start=2):
    note = "Demo: cross-participant ref. Replace with matched human transcription."
    for ci, val in enumerate(
        [pid, asheet, rsheet, MODEL_SIZE,
         round(w,4), round(c,4), f"{ag:.1f}%", f"{n}/30", note], 1):
        ws_s.cell(row=ri, column=ci, value=val)

for col_cells in ws_s.columns:
    max_len = max((len(str(cell.value or "")) for cell in col_cells), default=10)
    ws_s.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 60)

wb.save(output_path)
print(f"\nSaved: {output_path}")
files.download(output_path)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.3 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


Saving 038010_EIT-2A.mp3 to 038010_EIT-2A.mp3
Saving 038011_EIT-1A.mp3 to 038011_EIT-1A.mp3
Saving 038012_EIT-2A.mp3 to 038012_EIT-2A.mp3
Saving 038015_EIT-1A.mp3 to 038015_EIT-1A.mp3
Saving AutoEIT Sample Audio for Transcribing.xlsx to AutoEIT Sample Audio for Transcribing.xlsx
Saving AutoEIT Sample Transcriptions for Scoring.xlsx to AutoEIT Sample Transcriptions for Scoring.xlsx
  Audio loaded: 038010
  Audio loaded: 038011
  Audio loaded: 038012
  Audio loaded: 038015

Audio xlsx   : /content/autoeit/AutoEIT Sample Audio for Transcribing.xlsx
Scoring xlsx : /content/autoeit/AutoEIT Sample Transcriptions for Scoring.xlsx
  Preprocessed: 038010
  Preprocessed: 038011
  Preprocessed: 038012
  Preprocessed: 038015

Loading whisper medium on CPU...


100%|█████████████████████████████████████| 1.42G/1.42G [00:16<00:00, 93.5MiB/s]


Whisper medium ready

  Transcribing 038010...
  038010: 62 raw segments ✓

  Transcribing 038011...
  038011: 53 raw segments ✓

  Transcribing 038012...
  038012: 95 raw segments ✓

  Transcribing 038015...
  038015: 37 raw segments ✓
  Sheet matched: 038010 → '38010-2A'
  Sheet matched: 038011 → '38011-1A'
  Sheet matched: 038012 → '38012-2A'
  Sheet matched: 038015 → '38015-1A'
  WER pairing:  038010 → '38001-1A' (demo — different participant)
  WER pairing:  038011 → '38002-2A' (demo — different participant)
  WER pairing:  038012 → '38004-2A' (demo — different participant)
  WER pairing:  038015 → '38006-2A' (demo — different participant)
  038010: 30/30 sentences aligned
  038011: 30/30 sentences aligned
  038012: 30/30 sentences aligned
  038015: 30/30 sentences aligned
Computing WER against Scoring xlsx references (cross-participant demo)

PID        Audio Sheet    Ref Sheet          WER     CER   Agreement   Scored
038010     38010-2A       38001-1A         4.654   4.361     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>